In [1]:
import os

In [2]:
%pwd

'c:\\Users\\Ayush\\OneDrive\\Desktop\\new folder\\End-to-End-machine-learning-project-with-mlflow\\research'

In [3]:
os.chdir("../")

In [4]:
%pwd

'c:\\Users\\Ayush\\OneDrive\\Desktop\\new folder\\End-to-End-machine-learning-project-with-mlflow'

In [5]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class ModelTrainerConfig:
    root_dir: Path
    train_data_path: Path
    test_data_path: Path
    model_name: str
    alpha: float
    l1_ratio: float
    target_column: str

In [8]:
from mlProject.constants import *
from mlProject.utils.common import read_yaml, create_directories

In [13]:
class ConfigurationManager:
    def __init__(
        self,
        config_filepath= CONFIG_FILE_PATH,
        params_filepath= PARAMS_FILE_PATH,
        schema_filepath= SCHEMA_FILE_PATH
    ):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        self.schema = read_yaml(schema_filepath)

        create_directories([self.config.artifacts_root])

    def get_model_trainer_config(self) -> ModelTrainerConfig:
        config = self.config.model_trainer
        params = self.params.ElasticNet
        schema = self.schema.TARGET_COLUMN

        create_directories([config.root_dir])
        model_trainer_config = ModelTrainerConfig(
            root_dir=Path(config.root_dir),
            train_data_path=Path(config.train_data_path),
            test_data_path=Path(config.test_data_path),
            model_name=config.model_name,
            alpha=params.alpha,
            l1_ratio=params.l1_ratio,
            target_column=schema.name
        )

        return model_trainer_config

In [14]:
import pandas as pd
import os 
from mlProject import logger
from sklearn.linear_model import ElasticNet
import joblib


In [15]:
class ModelTrainer:
    def __init__(self, config: ModelTrainerConfig):
        self.config = config

    def train_model(self):
        logger.info("Loading training and testing data")
        train_data = pd.read_csv(self.config.train_data_path)
        test_data = pd.read_csv(self.config.test_data_path)

        logger.info("Splitting data into features and target")
        X_train = train_data.drop(self.config.target_column,axis=1)
        y_train = train_data[self.config.target_column]

        X_test = test_data.drop(self.config.target_column,axis=1)
        y_test = test_data[self.config.target_column]

        logger.info("Training the ElasticNet model")
        model = ElasticNet(alpha=self.config.alpha, l1_ratio=self.config.l1_ratio, random_state=42)
        model.fit(X_train, y_train)

        logger.info("Saving the trained model")
        joblib.dump(model, os.path.join(self.config.root_dir, self.config.model_name))

In [16]:
try:
    config_manager = ConfigurationManager()
    model_trainer_config = config_manager.get_model_trainer_config()
    model_trainer = ModelTrainer(config=model_trainer_config)
    model_trainer.train_model()
except Exception as e:
    raise e

[2026-03-29 04:57:30,510: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-03-29 04:57:30,512: INFO: common: yaml file: params.yaml loaded successfully]
[2026-03-29 04:57:30,517: INFO: common: yaml file: schema.yaml loaded successfully]
[2026-03-29 04:57:30,519: INFO: common: created directory at: artifacts]
[2026-03-29 04:57:30,521: INFO: common: created directory at: artifacts/model_trainer]
[2026-03-29 04:57:30,521: INFO: 3113094994: Loading training and testing data]
[2026-03-29 04:57:30,531: INFO: 3113094994: Splitting data into features and target]
[2026-03-29 04:57:30,537: INFO: 3113094994: Training the ElasticNet model]
[2026-03-29 04:57:30,549: INFO: 3113094994: Saving the trained model]
